# Mushroom Edible vs Poisonous Classifier

Trains a binary image classifier (edible / poisonous) on the Kaggle dataset
"Over 3000 images of edible and poisonous sporocarps" and exports a
quantized TFLite model for on-device inference in Pocket Medic.

**Dataset**: https://www.kaggle.com/datasets/ [edible-poisonous-mushroom-sporocarps]
**Target classes**: `edible`, `poisonous`
**Model**: MobileNetV2 (transfer learning, frozen backbone)
**Output**: `mushroom_classifier.tflite` + `labels.txt`

In [ ]:
# Install dependencies
!pip install -q tensorflow tensorflow-datasets matplotlib
import tensorflow as tf
print(tf.__version__)

## 1. Download and prepare the dataset

The dataset is a ZIP on Kaggle with two folders:
- `edible mushroom sporocarp/` (715 images)
- `poisonous mushroom sporocarp/` (remaining images)

Adjust the path below if you download it manually.

In [ ]:
import os
import zipfile
import shutil

# --- Option A: Download via Kaggle API (recommended) ---
# Place your kaggle.json in ~/.kaggle/ before running
# !kaggle datasets download -d [DATASET_SLUG] -p /content/
# !unzip -o /content/[ZIP_NAME].zip -d /content/mushroom_data/

# --- Option B: Point to a local unzipped folder ---
DATA_DIR = '/content/mushroom_data'
EDIBLE_DIR = os.path.join(DATA_DIR, 'edible mushroom sporocarp')
POISON_DIR = os.path.join(DATA_DIR, 'poisonous mushroom sporocarp')

# If the dataset is still zipped, unzip it
zip_path = '/content/mushroom_dataset.zip'
if os.path.exists(zip_path) and not os.path.isdir(DATA_DIR):
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/')

# Count images
def count_images(folder):
    if not os.path.isdir(folder):
        return 0
    return sum(1 for f in os.listdir(folder) if f.lower().endswith(('.jpg', '.jpeg', '.png')))

n_edible = count_images(EDIBLE_DIR)
n_poison = count_images(POISON_DIR)
print(f'Edible images: {n_edible}')
print(f'Poisonous images: {n_poison}')
print(f'Total: {n_edible + n_poison}')

## 2. Build tf.data pipelines

We use `image_dataset_from_directory` which expects this structure:
```
mushroom_data/
  edible mushroom sporocarp/  -> class 0 (edible)
  poisonous mushroom sporocarp/ -> class 1 (poisonous)
```

If the folder names differ, rename them first.

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10

# Create a consolidated folder with class subfolders for tf.keras utils
CONSOLIDATED = '/content/mushroom_consolidated'
os.makedirs(os.path.join(CONSOLIDATED, 'edible'), exist_ok=True)
os.makedirs(os.path.join(CONSOLIDATED, 'poisonous'), exist_ok=True)

def copy_images(src_dir, dst_subdir):
    os.makedirs(dst_subdir, exist_ok=True)
    for f in os.listdir(src_dir):
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            shutil.copy2(os.path.join(src_dir, f), os.path.join(dst_subdir, f))

copy_images(EDIBLE_DIR, os.path.join(CONSOLIDATED, 'edible'))
copy_images(POISON_DIR, os.path.join(CONSOLIDATED, 'poisonous'))

train_ds = tf.keras.utils.image_dataset_from_directory(
    CONSOLIDATED,
    validation_split=0.2,
    subset='training',
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='int',
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    CONSOLIDATED,
    validation_split=0.2,
    subset='validation',
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='int',
)

class_names = train_ds.class_names
print('Classes:', class_names)

# Prefetch for performance
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

## 3. Build transfer-learning model (MobileNetV2)

Freeze the backbone, add a classification head, train only the head.

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
)
base_model.trainable = False  # freeze backbone

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

model.summary()

## 4. Train

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2),
    ],
)

# Plot accuracy/loss
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'], label='train_acc')
axes[0].plot(history.history['val_accuracy'], label='val_acc')
axes[0].set_title('Accuracy')
axes[0].legend()
axes[1].plot(history.history['loss'], label='train_loss')
axes[1].plot(history.history['val_loss'], label='val_loss')
axes[1].set_title('Loss')
axes[1].legend()
plt.tight_layout()
plt.savefig('/content/training_history.png', dpi=150)
plt.show()

## 5. Fine-tune (optional, unfreeze top layers)

Unfreeze the last 20 layers of MobileNetV2 and train with a very low LR.

In [ ]:
# Unfreeze top layers
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 20
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy'],
)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True),
    ],
)

print(f'Final val accuracy: {max(history_fine.history["val_accuracy"]):.4f}')

## 6. Export quantized TFLite model

The quantized model is ~5-10 MB and runs efficiently on mobile.

In [ ]:
export_dir = '/content/output'
os.makedirs(export_dir, exist_ok=True)

# Full integer quantization for on-device
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

# Representative dataset for calibration
def representative_dataset():
    for images, _ in val_ds.take(50):
        yield [images.numpy().astype('float32')]

converter.representative_dataset = representative_dataset

tflite_model = converter.convert()

model_path = os.path.join(export_dir, 'mushroom_classifier.tflite')
with open(model_path, 'wb') as f:
    f.write(tflite_model)

print(f'Model size: {os.path.getsize(model_path) / 1024 / 1024:.2f} MB')

# Write labels file
labels_path = os.path.join(export_dir, 'labels.txt')
with open(labels_path, 'w') as f:
    for name in class_names:
        f.write(name + '\n')

print('Exported:', model_path)
print('Labels:', labels_path)

## 7. Verify the TFLite model

Run a quick inference check to make sure the model works.

In [ ]:
import numpy as np

interpreter = tf.lite.Interpreter(model_path=model_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print('Input:', input_details)
print('Output:', output_details)

# Test with a random image (shape check)
dummy_input = np.random.randint(0, 255, (1, IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
interpreter.set_tensor(input_details[0]['index'], dummy_input)
interpreter.invoke()
output = interpreter.get_tensor(output_details[0]['index'])
print('Output shape:', output.shape)
print('Raw output:', output)

# Decode prediction
predicted_class = np.argmax(output)
confidence = output[0][predicted_class]
print(f'Predicted: {class_names[predicted_class]} ({confidence:.2%})')

## 8. Download the model files

After training, download `mushroom_classifier.tflite` and `labels.txt`
to your Flutter project's `assets/` directory.

In [ ]:
# In Colab, download the files
from google.colab import files
files.download(model_path)
files.download(labels_path)